# Step 11: Qwen judge on qualitative claims, 3 tool orders (Google Colab)

Runs Qwen3-4B (4-bit) on the qualitative claims in the 100 Step 8 test traces, three times each with the tool results in a different order.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. Have `step11_traces_for_colab.zip` ready to upload. It is in `data/processed/agent/` in the repository folder on your machine and holds the 100 test traces (simulator data). The traces are git-ignored, so they are not on GitHub.

**What runs where.** The code cells that start with `%%writefile` write exact copies of the repository files (`src/agent/judge.py`, `audit.py`, `llm.py`, `scripts/run_judge.py`, `scripts/judge_agreement.py`) into this Colab session. Two small files are cut down for Colab and say so at the top. The model downloads from Hugging Face at a pinned revision.

**At the end** cell 9 downloads a zip of the results. Put its files in `reports/step_11/` in the repository.

In [ ]:
# 1. GPU check and packages
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U "transformers>=4.51" bitsandbytes accelerate
import os
os.makedirs("src/agent", exist_ok=True)
os.makedirs("scripts", exist_ok=True)
os.makedirs("reports/step_11", exist_ok=True)
os.makedirs("data/processed/agent/traces/smollm_untuned/test", exist_ok=True)
print("folders ready in", os.getcwd())

## 2. Project code
Run each `%%writefile` cell once. They only write files.

In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/agent/__init__.py


In [ ]:
%%writefile src/agent/review.py
"""Colab subset of src/agent/review.py: only the number pattern, which audit.py and judge.py import. The full file needs the simulator and policy code."""
import re

NUMBER = re.compile(r"(?<![\w.])-?\d+(?:,\d{3})*(?:\.\d+)?")


In [ ]:
%%writefile src/agent/audit.py
"""Rule-based audit of explanations against the evidence in a trace. Each rule is narrow, so the counts are a lower bound on errors."""
import re

from src.agent.review import NUMBER

NO_ORDERS = re.compile(r"no open orders?\b(?! that)|open orders? (?:is|are) empty|no pending orders?", re.I)
ORDER_CLAIM = re.compile(r"open orders? (?:is|are) (?:not expected|delayed|due|expected|pending|on track|at risk)|open orders? (?:may|that may)", re.I)
DELAYED = re.compile(r"open orders? (?:is|are) delayed", re.I)
HIGH_RISK = re.compile(r"high risk of stockout|stockout risk is (?:high|likely)|risk of stockout is high", re.I)
LOW_RISK = re.compile(r"low risk of stockout|stockout risk is low|risk of stockout is low", re.I)
RISK_NUMBER = re.compile(r"(?:stockout risk of|risk of stockout is|probability of stockout \(|stockout \()\s*(\d+(?:\.\d+)?)%", re.I)
EXCEEDS_STOCK = re.compile(r"(?:demand|forecast)[^.]*(?:exceeding|higher than)[^.]*(?:stock|on-hand)", re.I)
HIGH_RISK_LIMIT, LOW_RISK_LIMIT = 20.0, 50.0


def audit(explanation, evidence):
    """Return the name of each rule the explanation breaks."""
    stock = evidence.get("get_inventory_status", {})
    risk = evidence.get("get_stockout_risk", {})
    orders = evidence.get("get_open_orders", {})
    forecast = evidence.get("get_demand_forecast", {})
    suggestion = evidence.get("get_escalation_suggestion", {})
    risks = [value for value in (risk.get("tabpfn_risk_percent"), risk.get("lightgbm_risk_percent")) if value is not None]
    listed = orders.get("open_orders", [])
    found = []
    if listed and NO_ORDERS.search(explanation):
        found.append("says_no_open_orders_but_one_exists")
    if not listed and ORDER_CLAIM.search(explanation):
        found.append("describes_an_open_order_that_does_not_exist")
    if listed and DELAYED.search(explanation) and not orders.get("units_overdue"):
        found.append("says_delayed_but_nothing_overdue")
    if risks and HIGH_RISK.search(explanation) and max(risks) < HIGH_RISK_LIMIT:
        found.append("says_high_risk_but_both_models_under_20")
    if risks and LOW_RISK.search(explanation) and max(risks) >= LOW_RISK_LIMIT:
        found.append("says_low_risk_but_a_model_at_50_or_more")
    if risks:
        for text in RISK_NUMBER.findall(explanation):
            if not any(abs(float(text) - value) < 1 for value in risks):
                found.append("stockout_risk_number_not_from_either_model")
                break
    if EXCEEDS_STOCK.search(explanation) and forecast.get("forecast_units_next_14_days") is not None and stock.get("on_hand_units") is not None and forecast["forecast_units_next_14_days"] <= stock["on_hand_units"]:
        found.append("says_demand_exceeds_stock_but_it_does_not")
    if "conflicts with the escalation suggestion" in explanation and suggestion.get("suggested_category") == "human_review":
        found.append("says_it_conflicts_with_the_suggestion_it_followed")
    if not NUMBER.search(explanation):
        found.append("cites_no_numbers")
    return found


In [ ]:
%%writefile src/agent/judge.py
"""Step 11: check an agent explanation claim by claim against the tool results its run saw.

Code checks numbers, open-order claims and value bands. A local LLM checks qualitative claims, one at a time,
with the evidence and the project's definitions, and never sees the agent's category. Nothing here calls an API.
"""
import json
import random
import re
from dataclasses import asdict, dataclass, field

from src.agent.audit import DELAYED, NO_ORDERS, ORDER_CLAIM
from src.agent.review import NUMBER

TOOL_ORDER = ["get_inventory_status", "get_demand_forecast", "get_stockout_risk", "get_open_orders", "get_escalation_suggestion"]
ERROR_VERDICTS = {"wrong_value", "misattributed", "unsupported", "contradicted"}
WHOLE_TOLERANCE, DECIMAL_TOLERANCE = 1.0, 0.051

# Horizons such as "next 14 days" set the time window; they are not claims about a tool result.
HORIZON = re.compile(r"(?:next|last)[\s-]+(?:the[\s-]+)?(?:14|28|30)[\s-]+days?|over\s+(?:14|28)\s+days?", re.I)
SENTENCE_END = re.compile(r"(?<=[.!?])\s+(?=[A-Z'\"(])")

# Each stated number takes its unit from the words right after it, so "7 units" cannot bind to days of cover.
UNIT_AFTER = [(re.compile(r"\s*%"), "percent"), (re.compile(r"\s*points?\b", re.I), "percent"), (re.compile(r"(?:\s+|-)days?\b", re.I), "days"), (re.compile(r"\s+units?\b", re.I), "units")]

# Field groups. Each value is a pattern over field names, and the unit kinds the group accepts.
GROUPS = {
    "on_hand": (r"inventory\.on_hand_units", {"units"}),
    "on_order": (r"inventory\.on_order_units", {"units"}),
    "cover": (r"inventory\.days_of_cover", {"days"}),
    "order_count": (r"inventory\.open_orders|orders\.count", {"count"}),
    "sales_rate": (r"inventory\.sales_per_day_last_28_days", {"rate"}),
    "sold_out": (r"inventory\.sold_out_days_last_28_days", {"days"}),
    "forecast": (r"forecast\.forecast_units_next_14_days", {"units"}),
    "low_estimate": (r"forecast\.low_estimate_units", {"units"}),
    "high_estimate": (r"forecast\.high_estimate_units", {"units"}),
    "stockout": (r"risk\.(?:tabpfn|lightgbm)_risk_percent", {"percent"}),
    "tabpfn": (r"risk\.tabpfn_risk_percent", {"percent"}),
    "lightgbm": (r"risk\.lightgbm_risk_percent", {"percent"}),
    "gap": (r"risk\.models_differ_by_points", {"percent"}),
    "order_units": (r"orders\.open_orders\[\d+\]\.units|orders\.total_units|orders\.units_due_next_14_days|orders\.units_overdue", {"units"}),
    "order_days": (r"orders\.open_orders\[\d+\]\.days_until_promised|orders\.days_to_next_delivery|orders\.days_overdue", {"days"}),
    "delay_risk": (r"orders\.open_orders\[\d+\]\.delay_risk_percent", {"percent"}),
    "late_rate": (r"orders\.supplier_late_percent_(?:long_run|last_30_days)", {"percent"}),
    "late_long": (r"orders\.supplier_late_percent_long_run", {"percent"}),
    "late_recent": (r"orders\.supplier_late_percent_last_30_days", {"percent"}),
    "probability": (r"suggestion\.probabilities_percent\.\w+", {"percent"}),
    "prob_monitor": (r"suggestion\.probabilities_percent\.monitor", {"percent"}),
    "prob_reorder": (r"suggestion\.probabilities_percent\.reorder_recommendation", {"percent"}),
    "prob_expedite": (r"suggestion\.probabilities_percent\.expedite_recommendation", {"percent"}),
    "prob_human": (r"suggestion\.probabilities_percent\.human_review", {"percent"}),
    "value_band": (None, {"percent", "days", "units", "count", "rate"}),  # a band has no number: a number bound here stays unbound
}

# Words that name a field. Earlier patterns win where two overlap, so "probability of stockout" is not read as a classifier probability.
CUES = [(re.compile(pattern, re.I), group) for pattern, group in [
    (r"days? of cover", "cover"),
    (r"days? until (?:the )?promised(?: date)?", "order_days"),
    (r"(?:next )?delivery (?:is )?in|delivery window|due in|arrive in|arriving in", "order_days"),
    (r"sales per day", "sales_rate"),
    (r"sold[- ]out days", "sold_out"),
    (r"low estimate", "low_estimate"),
    (r"high estimate", "high_estimate"),
    (r"value band (?:of|\()", "value_band"),  # only a number written straight after the band, such as "value band of 18 units"
    (r"delay risk|risk of delay", "delay_risk"),
    (r"late rates?|late percent", "late_rate"),
    (r"differ(?:s|ing)? by|difference", "gap"),
    (r"(?:probability|chance|risk) of (?:a )?stockout|stockout (?:risk|probability|chance)", "stockout"),
    (r"probability of expedit\w*", "prob_expedite"),
    (r"probability of reorder\w*", "prob_reorder"),
    (r"probability", "probability"),
    (r"units on hand|on[- ]hand(?: units)?", "on_hand"),
    (r"units on order|on[- ]order(?: units)?", "on_order"),
    (r"units?(?= due\b)|units? overdue|total of", "order_units"),  # "units" alone, so "due in 9 days" stays a cue for the days
    (r"open orders?|orders? pending|pending orders?", "order_count"),
    (r"demand forecast|forecast(?:ed)?(?: demand)?|demand", "forecast"),
    (r"current inventory|inventory|stock(?: level)? is currently|current stock", "on_hand"),
    (r"\b(?:chance|risk)\b", "stockout"),
]]

# Words that narrow a group once it is chosen, such as which model a stockout risk came from.
QUALIFIERS = {
    "stockout": [(re.compile(r"tabpfn", re.I), "tabpfn"), (re.compile(r"lightgbm", re.I), "lightgbm")],
    "late_rate": [(re.compile(r"long[- ]run", re.I), "late_long"), (re.compile(r"last[- ]30[- ]days|recent", re.I), "late_recent")],
    "probability": [(re.compile(r"monitor", re.I), "prob_monitor"), (re.compile(r"reorder", re.I), "prob_reorder"), (re.compile(r"expedit", re.I), "prob_expedite"),
                    (re.compile(r"human[ _]review", re.I), "prob_human")],
}
BOTH_MODELS = re.compile(r"\bboth models\b", re.I)
RESPECTIVELY = re.compile(r"\brespectively\b", re.I)
SEPARATOR = re.compile(r",|;|\band\b|\bwhile\b", re.I)
ATTACHED_AFTER = 25  # characters allowed between a number and a qualifier written after it, as in "5.3% for the last 30 days"

ORDER_EXISTS = re.compile("|".join([ORDER_CLAIM.pattern, r"there (?:is|are) (?:an? |\d+ )?open orders?", r"\bopen orders? (?:of|has|have|show|shows)\b", r"\b\d+ open orders?",
                                    r"\bopen orders? (?:is|are) (?:also|\d)", r"\ban open order\b", r"\b\d+ orders? pending"]), re.I)
BAND_CLAIM = re.compile(r"(?<!below the )(?<!above the )(?<!within the )\b(low|medium|high) value band\b", re.I)

# Qualitative claims go to the LLM. Phrases checked elsewhere, or that describe the agent's own uncertainty, are removed first.
NOT_QUALITATIVE = re.compile(r"low estimate|high estimate|late rates?|delay risk|(?:(?:low|medium|high) )?value band|(?:high|low|medium) (?:in )?uncertainty(?: level)?|uncertainty is \w+|medium unsettled|"
                             r"high (?:probability|confidence)(?= (?:of )?\(?\d)|with a high probability(?= of \d)", re.I)
QUALITATIVE = re.compile(r"\b(?:high|higher|low|lower|medium|moderate|likely|unlikely|sufficient|insufficient|enough|concern|exceed\w*|below|above|within|over the current|"
                         r"on track|in time|too late|late|delay|delays|delayed|approaching|urgen\w*|need|needed|conflicts?|strongly|warrant|not zero|relatively|empty|zero)\b", re.I)

DEFINITIONS = """Definitions used in this project (project choices, not measured thresholds):
- Stockout risk is the chance of a stockout in the next 14 days, given by two models, TabPFN and LightGBM, in percent.
- High or likely stockout risk: at least one model at 50% or more. Low stockout risk: both models under 20%. Between 20% and 50% is neither clearly high nor clearly low.
- Supplier late rates under 10% are low. 10% or more is not low.
- An open order is delayed only if it has units overdue. Delay risk is the chance an open order arrives after its promised date.
- Stock covers the forecast when on-hand units are at least the 14-day forecast."""

JUDGE_SYSTEM = f"""You check one claim from an explanation of an inventory review against the tool results that review saw.
Use only the tool results and the definitions. Do not use outside knowledge.
{DEFINITIONS}
Answer "supported" if the tool results show the claim is true, "contradicted" if they show it is false, and "unclear" if they cannot settle it.
Reply with JSON only, in this form: {{"verdict": "supported" or "contradicted" or "unclear", "reason": "one sentence"}}"""
LLM_VERDICTS = {"supported", "contradicted", "unclear"}


@dataclass
class Claim:
    case_id: str
    sentence: int
    clause_index: int
    clause: str
    kind: str  # number, open_orders, value_band, qualitative
    text: str
    sentence_text: str = ""
    stated: float | None = None
    unit: str | None = None
    group: str | None = None
    require_all: bool = False
    fields: list = field(default_factory=list)
    actual: list = field(default_factory=list)
    verdict: str = ""
    detail: str = ""
    position: int = 0  # order within the explanation, so each claim ID is unique

    @property
    def claim_id(self):
        return f"{self.case_id}-{self.position:02d}"

    def row(self):
        return {"claim_id": self.claim_id, **{k: v for k, v in asdict(self).items() if k not in ("sentence_text", "position")},
                "fields": ";".join(self.fields), "actual": ";".join(map(str, self.actual))}


# Splitting ------------------------------------------------------------------------------------------------------------

def sentences(text):
    return [part.strip() for part in SENTENCE_END.split(text.strip()) if part.strip()]


def clauses(sentence):
    """Split a sentence at commas and semicolons outside parentheses."""
    parts, depth, start = [], 0, 0
    for position, char in enumerate(sentence):
        depth += (char == "(") - (char == ")")
        if depth == 0 and char in ",;" and sentence[position + 1:position + 2] == " " and not sentence[position + 2:].lower().startswith("respectively"):
            parts.append(sentence[start:position])
            start = position + 1
    parts.append(sentence[start:])
    return [part.strip().rstrip(".").strip() for part in parts if part.strip().rstrip(".").strip()]


def stated_numbers(clause):
    """Return (text, value, start, end, unit) for each number that is not part of a horizon."""
    horizons = [match.span() for match in HORIZON.finditer(clause)]
    found = []
    for match in NUMBER.finditer(clause):
        start, end = match.span()
        if any(a <= start and end <= b for a, b in horizons):
            continue
        rest = clause[end:]
        unit = next((kind for pattern, kind in UNIT_AFTER if pattern.match(rest)), None)
        found.append((match.group(), float(match.group().replace(",", "")), start, end, unit))
    return found


def cue_spans(clause):
    """Find field cues, keeping the earlier pattern where two overlap."""
    kept, text = [], clause.replace("_", " ")  # raw field names such as delay_risk_percent count as words
    for pattern, group in CUES:
        for match in pattern.finditer(text):
            if not any(match.start() < b and a < match.end() for a, b, _ in kept):
                kept.append((match.start(), match.end(), group))
    return sorted(kept)


def compatible(group, unit):
    if group == "order_count":
        return unit in (None, "units", "days", "percent")
    return unit is None or unit in GROUPS[group][1]


def resolve_count_cue(group, unit):
    """"open order" names a count, but "open order of 18 units" names units and "open order due in 5 days" names days."""
    if group != "order_count":
        return group
    return {"units": "order_units", "days": "order_days", "percent": "delay_risk"}.get(unit, group)


def nearest(spans, start, end, clause=None):
    """Pick the span closest to [start, end), ranking spans across a comma last; ties go to the span before."""
    best = None
    for a, b, *rest in spans:
        if a < end and start < b:
            continue
        distance = start - b if b <= start else a - end
        between = (clause[b:start] if b <= start else clause[end:a]) if clause else ""
        key = ("," in between or ";" in between, distance, 0 if b <= start else 1)
        if best is None or key < best[0]:
            best = (key, (a, b, *rest))
    return best[1] if best else None


def qualify(group, clause, numbers, index):
    """Narrow a group with a qualifier. In order: a "respectively" list, a qualifier attached after the number ("5.3% last 30 days"),
    one attached before it ("tabpfn risk of 2.2%"), then the nearest in the clause. Attached means no comma, "and" or "while" between."""
    options = QUALIFIERS.get(group)
    if not options:
        return group, False
    text = clause.replace("_", " ")
    marks = sorted((m.start(), m.end(), refined) for pattern, refined in options for m in pattern.finditer(text))
    start, end = numbers[index][2], numbers[index][3]
    if RESPECTIVELY.search(clause):
        same = [n for n in numbers if n[5] == group]
        position = [n[2] for n in same].index(start)
        after = [m for m in marks if m[0] >= same[-1][3]]
        if len(after) == len(same):
            return after[position][2], False
    next_start = numbers[index + 1][2] if index + 1 < len(numbers) else len(clause)
    previous_end = numbers[index - 1][3] if index > 0 else 0
    after = [m for m in marks if end <= m[0] and m[1] <= next_start and m[0] - end <= ATTACHED_AFTER and not SEPARATOR.search(text[end:m[0]])]
    if after:
        return after[0][2], False
    before = [m for m in marks if previous_end <= m[0] and m[1] <= start and not SEPARATOR.search(text[m[1]:start])]
    if before:
        return before[-1][2], False
    if marks:
        return nearest(marks, start, end)[2], False
    return group, group == "stockout" and bool(BOTH_MODELS.search(clause))


def bind_numbers(clause):
    """Return (text, value, unit, group or None, require_all) for each stated number in a clause."""
    numbers = stated_numbers(clause)
    cues = cue_spans(clause)
    chosen = []
    for text, value, start, end, unit in numbers:
        usable = [c for c in cues if compatible(c[2], unit)]
        cue = nearest(usable, start, end, clause)
        chosen.append((text, value, start, end, unit, resolve_count_cue(cue[2], unit) if cue else None))
    bound = []
    for index, (text, value, start, end, unit, group) in enumerate(chosen):
        require_all = False
        if group is not None:
            group, require_all = qualify(group, clause, chosen, index)
        if group == "value_band":
            group = None
        bound.append((text, value, unit, group, require_all))
    return bound


def extract_claims(case_id, explanation):
    """Split an explanation into claims. Needs no evidence, so coverage can be measured on any explanation."""
    claims = []
    for s_index, sentence in enumerate(sentences(explanation)):
        for c_index, clause in enumerate(clauses(sentence)):
            base = dict(case_id=case_id, sentence=s_index, clause_index=c_index, clause=clause, sentence_text=sentence)
            for text, value, unit, group, require_all in bind_numbers(clause):
                claims.append(Claim(**base, kind="number", text=text, stated=value, unit=unit, group=group, require_all=require_all))
            order_text = ""
            for pattern, label in [(NO_ORDERS, "no open orders"), (DELAYED, "open orders are delayed"), (ORDER_EXISTS, "an open order exists")]:
                match = pattern.search(clause)
                if match:
                    claims.append(Claim(**base, kind="open_orders", text=label))
                    order_text = match.group()
                    break
            band = BAND_CLAIM.search(clause)
            if band:
                claims.append(Claim(**base, kind="value_band", text=band.group(1).lower()))
            rest = NOT_QUALITATIVE.sub(" ", clause.replace(order_text, " ") if order_text else clause)
            if QUALITATIVE.search(rest):
                claims.append(Claim(**base, kind="qualitative", text=clause))
    for position, claim in enumerate(claims):
        claim.position = position
    return claims


# Code verdicts --------------------------------------------------------------------------------------------------------

def flatten(evidence):
    """Name every numeric field in the evidence, plus the order count and total ordered units."""
    names = {"get_inventory_status": "inventory", "get_demand_forecast": "forecast", "get_stockout_risk": "risk", "get_open_orders": "orders", "get_escalation_suggestion": "suggestion"}
    flat = {}

    def walk(prefix, value):
        if isinstance(value, bool):
            return
        if isinstance(value, dict):
            for key, item in value.items():
                walk(f"{prefix}.{key}", item)
        elif isinstance(value, list):
            for position, item in enumerate(value):
                walk(f"{prefix}[{position}]", item)
        elif value is None or isinstance(value, (int, float)):
            flat[prefix] = None if value is None else float(value)

    for tool, short in names.items():
        if tool in evidence:
            walk(short, {k: v for k, v in evidence[tool].items() if k != "missing_fields"})
    listed = evidence.get("get_open_orders", {}).get("open_orders")
    if isinstance(listed, list):
        flat["orders.count"] = float(len(listed))
        flat["orders.total_units"] = float(sum(item.get("units") or 0 for item in listed))
    return flat


def matches(stated_text, stated, actual):
    """A whole number may round or truncate the actual value. A decimal must match to its last digit."""
    if actual is None:
        return False
    tolerance = DECIMAL_TOLERANCE if "." in stated_text else WHOLE_TOLERANCE
    return abs(stated - actual) < tolerance if tolerance == WHOLE_TOLERANCE else abs(stated - actual) <= tolerance


def check_number(claim, flat):
    """Compare a stated number with the field it names. Only a non-zero decimal can be misattributed: a whole number, or zero, can equal another field by chance."""
    elsewhere = [name for name, value in flat.items() if matches(claim.text, claim.stated, value)]
    if claim.group is None:
        claim.verdict = "unverified" if elsewhere else "unsupported"
        claim.detail = ("in " + ",".join(elsewhere)) if elsewhere else "in no tool result"
        return claim
    pattern = re.compile(GROUPS[claim.group][0])
    named = [name for name in flat if pattern.fullmatch(name)]
    claim.fields = named
    claim.actual = [flat[name] for name in named]
    present = [name for name in named if flat[name] is not None]
    if not present:
        claim.verdict, claim.detail = "unsupported", "named field is missing from the tool results"
        return claim
    hits = [name for name in present if matches(claim.text, claim.stated, flat[name])]
    if (claim.require_all and len(hits) == len(present)) or (not claim.require_all and hits):
        claim.verdict = "supported"
        return claim
    others = [name for name in elsewhere if name not in named]
    if others and "." in claim.text and claim.stated != 0:
        claim.verdict, claim.detail = "misattributed", "value of " + ",".join(others)
    else:
        claim.verdict = "wrong_value"
        claim.detail = "both models must match" if claim.require_all and hits else ""
    return claim


def check_orders(claim, evidence):
    orders = evidence.get("get_open_orders")
    if orders is None:
        claim.verdict, claim.detail = "unsupported", "open orders tool result missing"
        return claim
    listed = orders.get("open_orders") or []
    if claim.text == "no open orders":
        truth = not listed
    elif claim.text == "open orders are delayed":
        truth = bool(listed) and bool(orders.get("units_overdue"))
    else:
        truth = bool(listed)
    claim.verdict = "supported" if truth else "contradicted"
    claim.detail = f"{len(listed)} listed, {orders.get('units_overdue')} units overdue"
    return claim


def check_band(claim, evidence):
    band = evidence.get("get_inventory_status", {}).get("value_band")
    claim.verdict = "unsupported" if band is None else ("supported" if band == claim.text else "contradicted")
    claim.detail = f"value band is {band}"
    return claim


# LLM verdicts ---------------------------------------------------------------------------------------------------------

def tool_orders(count, seed=11):
    """The fixed order first, then distinct shuffled orders, so position bias can be measured."""
    found, generator = [list(TOOL_ORDER)], random.Random(seed)
    while len(found) < count:
        order = list(TOOL_ORDER)
        generator.shuffle(order)
        if order not in found:
            found.append(order)
    return found


def judge_prompt(claim, evidence, order=TOOL_ORDER):
    """Messages for one claim. The prompt holds the claim, its sentence and the tool results; never the agent's category."""
    results = "\n".join(f"{tool}: {json.dumps(evidence[tool])}" for tool in order if tool in evidence)
    user = f"Tool results:\n{results}\n\nSentence the claim comes from: \"{claim.sentence_text}\"\nClaim to check, only this part: \"{claim.text}\""
    return [{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}]


def parse_reply(text):
    """Accept a JSON object, optionally in a code fence, and nothing else. Return (verdict, reason) or None."""
    body = text.strip()
    fence = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", body, re.S)
    if fence:
        body = fence.group(1)
    try:
        reply = json.loads(body)
    except json.JSONDecodeError:
        return None
    if not isinstance(reply, dict) or reply.get("verdict") not in LLM_VERDICTS:
        return None
    return reply["verdict"], str(reply.get("reason", ""))


def judge_qualitative(claim, evidence, llm, orders=(TOOL_ORDER,)):
    """Ask the LLM once per tool order. The claim's verdict is the one from the fixed order; changes across orders are recorded."""
    verdicts = []
    for order in orders:
        reply = llm.generate(judge_prompt(claim, evidence, order), tools=None)
        parsed = parse_reply(reply["text"])
        verdicts.append("judge_error" if parsed is None else parsed[0])
        if len(verdicts) == 1:
            claim.detail = reply["text"].strip()[:300] if parsed is None else parsed[1]
    claim.verdict = verdicts[0]
    claim.actual = verdicts
    return claim


def judge_explanation(case_id, explanation, evidence, llm=None, orders=1):
    """Return every claim in one explanation with a verdict. Without an LLM, qualitative claims are not_checked."""
    flat = flatten(evidence)
    order_list = tool_orders(orders)
    claims = extract_claims(case_id, explanation)
    for claim in claims:
        if claim.kind == "number":
            check_number(claim, flat)
        elif claim.kind == "open_orders":
            check_orders(claim, evidence)
        elif claim.kind == "value_band":
            check_band(claim, evidence)
        elif llm is None:
            claim.verdict = "not_checked"
        else:
            judge_qualitative(claim, evidence, llm, order_list)
    return claims


def summarize(case_id, claims):
    errors = [c for c in claims if c.verdict in ERROR_VERDICTS]
    return {"case_id": case_id, "claims": len(claims), "errors": len(errors), "has_error": bool(errors),
            "code_error": any(c.kind != "qualitative" for c in errors), "error_claims": " | ".join(f"{c.verdict}: {c.text} ({c.group or c.kind})" for c in errors)}


In [ ]:
%%writefile src/agent/llm.py
"""Load a local instruction model with 4-bit weights and generate one reply at a time. Nothing leaves the machine."""
import copy
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODELS = {"qwen": ("Qwen/Qwen3-4B-Instruct-2507", "cdbee75f17c01a7cc42f958dc650907174af0554"), "smollm": ("HuggingFaceTB/SmolLM3-3B", "a07cc9a04f16550a088caea529712d1d335b0ac1")}


class LocalLLM:
    def __init__(self, name="smollm", max_new_tokens=400, max_input_tokens=3000):
        self.repo, self.revision = MODELS[name]
        self.max_new_tokens, self.max_input_tokens = max_new_tokens, max_input_tokens
        self.tokenizer = AutoTokenizer.from_pretrained(self.repo, revision=self.revision)
        quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
        self.model = AutoModelForCausalLM.from_pretrained(self.repo, revision=self.revision, quantization_config=quantization, device_map={"": 0}, dtype=torch.float16)
        self.model.eval()
        self.prefix_ids, self.prefix_cache = None, None

    def _template(self, messages, tools):
        return self.tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    def _prepare_prefix(self, messages, tools):
        """Save the work on the instructions every prompt shares, because re-reading them took most of each turn."""
        first, second = [self.tokenizer(self._template([messages[0], {"role": "user", "content": word}], tools)).input_ids for word in ("first", "second")]
        length = next(i for i, (a, b) in enumerate(zip(first, second)) if a != b)
        self.prefix_ids = first[:length]
        with torch.no_grad():
            self.prefix_cache = self.model(input_ids=torch.tensor([self.prefix_ids], device=self.model.device), use_cache=True).past_key_values

    def generate(self, messages, tools, use_prefix=True):
        """Return greedy output, so a repeat run gives the same text."""
        inputs = self.tokenizer(self._template(messages, tools), return_tensors="pt").to(self.model.device)
        length = inputs.input_ids.shape[1]
        if length > self.max_input_tokens:
            raise ValueError(f"prompt of {length} tokens is over the limit of {self.max_input_tokens}")
        started = time.time()
        extra = {}
        if use_prefix:
            if self.prefix_ids is None:
                self._prepare_prefix(messages, tools)
            if inputs.input_ids[0, :len(self.prefix_ids)].tolist() == self.prefix_ids:
                extra["past_key_values"] = copy.deepcopy(self.prefix_cache)
        with torch.no_grad():
            output = self.model.generate(**inputs, **extra, max_new_tokens=self.max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None, pad_token_id=self.tokenizer.eos_token_id)
        created = output[0][length:]
        return {"text": self.tokenizer.decode(created, skip_special_tokens=True), "input_tokens": int(length), "output_tokens": int(len(created)), "seconds": time.time() - started}


In [ ]:
%%writefile scripts/parse_trace_file.py
"""Colab stub: run_judge.py imports parse() for --source report, which needs the committed Step 8 report. Use --source traces here."""


def parse(*args, **kwargs):
    raise RuntimeError("--source report is not available in Colab; use --source traces")


In [ ]:
%%writefile scripts/run_judge.py
"""Judge agent explanations claim by claim and compare with the Step 8 audit. Runs locally; calls no API.

--source traces   all 100 test traces in data/processed (git-ignored)
--source report   the 10 complete traces committed in reports/step_08/test_traces.txt
--llm none        code layer only; qualitative claims are not_checked
--llm qwen        Qwen3-4B judges qualitative claims (needs a GPU). The agent was SmolLM3, so the default judge is a different family.
--orders K        ask the LLM K times per claim with tool results in different orders, to measure position bias
"""
import argparse
import json
import random
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import pandas as pd

from scripts.parse_trace_file import parse
from src.agent.audit import audit
from src.agent.judge import ERROR_VERDICTS, judge_explanation, summarize

TRACES = ROOT / "data/processed/agent/traces/smollm_untuned/test"
OUT = ROOT / "reports/step_11"
LABEL_COLUMNS = ["claim_id", "case_id", "kind", "text", "clause", "group", "stated"]


def load(source):
    if source == "report":
        complete, _ = parse()
        return [(c["case_id"], c["explanation"], c["evidence"], c["final"]) for c in complete.values()]
    rows = []
    for path in sorted(TRACES.glob("test-*.json")):
        trace = json.loads(path.read_text(encoding="utf-8"))
        rows.append((trace["case_id"], trace["recommendation"]["explanation"], trace["evidence"], trace["review"]["final_category"]))
    return rows


def label_sample(claims, size, seed):
    """All flagged claims plus a random sample of the rest, shuffled, with the judge's verdict left out so labels are blind."""
    flagged = claims[claims.verdict.isin(ERROR_VERDICTS)]
    rest = claims.drop(flagged.index)
    picked = pd.concat([flagged, rest.sample(min(size, len(rest)), random_state=seed)])
    picked = picked.sample(frac=1, random_state=seed)[LABEL_COLUMNS].assign(label="", note="")
    return picked, len(rest)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--source", choices=["traces", "report"], default="report")
    parser.add_argument("--llm", choices=["none", "qwen", "smollm"], default="none")
    parser.add_argument("--orders", type=int, default=1)
    parser.add_argument("--sample", type=int, default=100)
    parser.add_argument("--seed", type=int, default=11)
    args = parser.parse_args()
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    pd.set_option("display.width", 200, "display.max_colwidth", 90, "display.max_rows", 500)
    tag = f"{args.source}_{args.llm}"
    llm = None
    if args.llm != "none":
        from src.agent.llm import LocalLLM
        llm = LocalLLM(args.llm, max_new_tokens=120)
    OUT.mkdir(parents=True, exist_ok=True)

    cases = load(args.source)
    claims, explanations = [], []
    for case_id, explanation, evidence, final in cases:
        found = judge_explanation(case_id, explanation, evidence, llm=llm, orders=args.orders)
        claims += [c.row() for c in found]
        rules = [name for name in audit(explanation, evidence) if name != "cites_no_numbers"]
        explanations.append({**summarize(case_id, found), "final": final, "step8_rules": "; ".join(rules), "step8_error": bool(rules), "explanation": explanation})
    claims = pd.DataFrame(claims)
    explanations = pd.DataFrame(explanations)

    print(f"source {args.source}, judge {args.llm}, orders {args.orders}: {len(explanations)} explanations, {len(claims)} claims in {claims.groupby(['case_id', 'sentence', 'clause_index']).ngroups} clauses")
    print("\nCLAIMS BY KIND AND VERDICT\n", pd.crosstab(claims.kind, claims.verdict, margins=True, margins_name="all").to_string())
    print(f"\nexplanations with a factual error: judge {int(explanations.has_error.sum())} (code layer {int(explanations.code_error.sum())}), Step 8 rules {int(explanations.step8_error.sum())}")
    print("\nPER EXPLANATION, where either finds an error\n", explanations[explanations.has_error | explanations.step8_error][["case_id", "final", "error_claims", "step8_rules"]].to_string(index=False))
    missed = explanations[explanations.step8_error & ~explanations.code_error]
    print(f"\nStep 8 rule hits with no code-layer error: {len(missed)}")
    if len(missed):
        print(missed[["case_id", "step8_rules", "error_claims"]].to_string(index=False))
    new = explanations[explanations.code_error & ~explanations.step8_error]
    print(f"code-layer errors no Step 8 rule found: {len(new)} explanations ({', '.join(new.case_id)})")
    monitor = explanations[explanations.has_error & (explanations.final == "monitor")]
    print(f"errors on a final monitor case, which no person reviews: {len(monitor)} ({', '.join(monitor.case_id)})")
    print("\nFLAGGED CLAIMS\n", claims[claims.verdict.isin(ERROR_VERDICTS)][["case_id", "kind", "text", "group", "actual", "verdict", "detail"]].to_string(index=False))
    if args.orders > 1 and llm is not None:
        asked = claims[claims.kind == "qualitative"]
        changed = asked.actual.map(lambda text: len(set(text.split(";"))) > 1)
        print(f"\nPOSITION BIAS: {int(changed.sum())} of {len(asked)} qualitative verdicts change when only the order of tool results changes")
        print(asked[changed][["case_id", "text", "actual"]].to_string(index=False))

    claims.to_csv(OUT / f"claims_{tag}.csv", index=False)
    explanations.drop(columns="explanation").to_csv(OUT / f"explanations_{tag}.csv", index=False)
    sample, rest = label_sample(claims, args.sample, args.seed)
    sample.to_csv(OUT / f"claims_to_label_{tag}.csv", index=False)
    (OUT / f"label_sample_{tag}.json").write_text(json.dumps({"flagged": int(claims.verdict.isin(ERROR_VERDICTS).sum()), "unflagged_total": rest, "unflagged_sampled": min(args.sample, rest), "seed": args.seed}), encoding="utf-8")
    print(f"\nwrote claims_{tag}.csv, explanations_{tag}.csv, claims_to_label_{tag}.csv ({len(sample)} claims, judge verdict hidden) to {OUT.relative_to(ROOT)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/judge_agreement.py
"""Score the judge against hand labels. Calls no API.

Fill the `label` column of reports/step_11/claims_to_label_<tag>.csv with: error, ok, or skip (cannot tell). That file has no judge verdict,
so labels are blind. The sample is every flagged claim plus a random sample of the rest, so unflagged rows are weighted up to the
number they stand for when recall and kappa are computed.
"""
import argparse
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.agent.judge import ERROR_VERDICTS

OUT = ROOT / "reports/step_11"
LABELS = {"error", "ok"}


def cohen_kappa(judge, human, weight):
    """Weighted Cohen's kappa for two yes/no raters."""
    total = weight.sum()
    observed = weight[judge == human].sum() / total
    p_judge, p_human = weight[judge].sum() / total, weight[human].sum() / total
    expected = p_judge * p_human + (1 - p_judge) * (1 - p_human)
    return float("nan") if expected == 1 else (observed - expected) / (1 - expected)


def score(frame):
    """Precision uses every flagged claim, all of which are labelled. Recall and kappa weight unflagged claims up to their population."""
    judge, human, weight = frame.flagged.to_numpy(), (frame.label == "error").to_numpy(), frame.weight.to_numpy()
    true_positive = weight[judge & human].sum()
    flagged, errors = weight[judge].sum(), weight[human].sum()
    return {"labelled": len(frame), "flagged": int(judge.sum()), "human_errors": int(human.sum()),
            "precision": true_positive / flagged if flagged else np.nan, "recall_weighted": true_positive / errors if errors else np.nan,
            "kappa_weighted": cohen_kappa(judge, human, weight)}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--tag", required=True, help="such as traces_qwen")
    args = parser.parse_args()
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    claims = pd.read_csv(OUT / f"claims_{args.tag}.csv")
    labels = pd.read_csv(OUT / f"claims_to_label_{args.tag}.csv").fillna({"label": ""})
    sample = json.loads((OUT / f"label_sample_{args.tag}.json").read_text(encoding="utf-8"))
    labels["label"] = labels.label.str.strip().str.lower()
    unknown = sorted(set(labels.label) - LABELS - {"", "skip"})
    if unknown:
        raise SystemExit(f"unknown labels {unknown}; use error, ok or skip")
    frame = labels[labels.label.isin(LABELS)].merge(claims[["claim_id", "verdict"]], on="claim_id", validate="one_to_one")
    frame["flagged"] = frame.verdict.isin(ERROR_VERDICTS)
    frame["weight"] = np.where(frame.flagged, 1.0, sample["unflagged_total"] / max(sample["unflagged_sampled"], 1))
    print(f"labels: {len(frame)} usable, {int((labels.label == 'skip').sum())} skipped, {int((labels.label == '').sum())} blank")
    missing = int((labels.label == "").sum())
    if missing:
        print("WARNING: blank labels remain, so these numbers are not final")
    rows = {"all": score(frame), "code layer": score(frame[frame.kind != "qualitative"])}
    qualitative = frame[frame.kind == "qualitative"]
    if len(qualitative):
        rows["LLM layer"] = score(qualitative)
        print("LLM verdicts among labelled qualitative claims:", qualitative.verdict.value_counts().to_dict())
    print(pd.DataFrame(rows).T.to_string(float_format=lambda value: f"{value:.3f}"))
    print("\nDISAGREEMENTS\n", frame[frame.flagged != (frame.label == "error")][["claim_id", "kind", "text", "verdict", "label", "note"]].to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
# 3. Upload step11_traces_for_colab.zip and unpack the 100 traces
import zipfile
from pathlib import Path
from google.colab import files

uploaded = files.upload()
name = next(iter(uploaded))
target = Path("data/processed/agent/traces/smollm_untuned/test")
with zipfile.ZipFile(name) as archive:
    for member in archive.namelist():
        if member.endswith(".json"):
            (target / Path(member).name).write_bytes(archive.read(member))
traces = sorted(target.glob("test-*.json"))
print(len(traces), "traces, first", traces[0].name, "last", traces[-1].name)
assert len(traces) == 100, "expected 100 traces"

In [ ]:
# 4. Code layer check: no LLM. This must match the local run (27 explanations with a code-layer error, 741 claims).
!python scripts/run_judge.py --source traces --llm none | head -12

In [ ]:
# 5. Smoke test: judge one case's qualitative claims once each, time the calls, estimate the full run
import json, sys, time
import torch
sys.path.insert(0, ".")
from src.agent.llm import LocalLLM
from src.agent.judge import TOOL_ORDER, extract_claims, judge_qualitative

llm = LocalLLM("qwen", max_new_tokens=120)
trace = json.loads(Path("data/processed/agent/traces/smollm_untuned/test/test-0013.json").read_text(encoding="utf-8"))
claims = [c for c in extract_claims("test-0013", trace["recommendation"]["explanation"]) if c.kind == "qualitative"]
seconds = []
for claim in claims:
    started = time.time()
    judge_qualitative(claim, trace["evidence"], llm, [TOOL_ORDER])
    seconds.append(time.time() - started)
    print(f"{seconds[-1]:5.1f}s  {claim.verdict:13} {claim.text!r}\n        reason: {claim.detail}")
per_call = sum(seconds) / len(seconds)
print(f"\nmean {per_call:.1f}s per call; the full run is about 582 calls, so roughly {582 * per_call / 60:.0f} minutes")
print("peak GPU GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))

In [ ]:
# 6. Free the GPU before the full run, which loads its own copy of the model
import gc
del llm
gc.collect()
torch.cuda.empty_cache()
print("free GB:", round(torch.cuda.mem_get_info()[0] / 1e9, 2))

In [ ]:
# 7. Full run: all 100 traces, Qwen judge, 3 tool orders. Output streams here and to reports/step_11/run_traces_qwen.log
import subprocess
command = [sys.executable, "scripts/run_judge.py", "--source", "traces", "--llm", "qwen", "--orders", "3"]
started = time.time()
with open("reports/step_11/run_traces_qwen.log", "w", encoding="utf-8") as log, subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace") as process:
    for line in process.stdout:
        print(line, end="")
        log.write(line)
print(f"\nexit code {process.returncode}, {(time.time() - started) / 60:.1f} minutes")

In [ ]:
# 8. Sanity checks on the LLM layer. For inspection only: do not report LLM counts until the hand labels are scored.
import pandas as pd
pd.set_option("display.max_colwidth", 120)
result = pd.read_csv("reports/step_11/claims_traces_qwen.csv")
asked = result[result.kind == "qualitative"]
per_order = asked.actual.str.split(";")
print("qualitative claims:", len(asked))
print("verdict in the fixed order:", asked.verdict.value_counts().to_dict())
print("judge_error in any order:", int(per_order.map(lambda v: "judge_error" in v).sum()))
print("verdict changes across orders:", int(per_order.map(lambda v: len(set(v)) > 1).sum()))
asked[["case_id", "text", "actual", "detail"]].head(20)

In [ ]:
# 9. Download the results. Put these files in reports/step_11/ in the repository.
import shutil
names = ["run_traces_qwen.log", "claims_traces_qwen.csv", "explanations_traces_qwen.csv", "claims_to_label_traces_qwen.csv", "label_sample_traces_qwen.json"]
with zipfile.ZipFile("step11_qwen_results.zip", "w") as archive:
    for name in names:
        archive.write(f"reports/step_11/{name}", name)
files.download("step11_qwen_results.zip")

## Next: hand labels

Open `claims_to_label_traces_qwen.csv` from the results zip. It has no judge verdict, so labels are blind. For each row, fill `label` with `error`, `ok` or `skip` (cannot tell), using the tool results in that case's trace. Optionally add a `note`.

Then score the labels, either on your machine with `python scripts/judge_agreement.py --tag traces_qwen`, or here with cell 10 if this session is still open.

In [ ]:
# 10. Optional, in the same session: upload the labelled claims_to_label_traces_qwen.csv and score the judge
uploaded = files.upload()
Path("reports/step_11/claims_to_label_traces_qwen.csv").write_bytes(next(iter(uploaded.values())))
!python scripts/judge_agreement.py --tag traces_qwen